In [ ]:
# Imports
from pathlib import Path
import numpy as np
import cv2
from itertools import cycle
import pandas as pd
import bokeh
from matplotlib import rcParams
from eye_tracking_system_tools.preprocessing import BlockSync, utility_functions as uf

rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded and editable
rcParams['ps.fonttype'] = 42  # Ensure compatibility with vector outputs
%matplotlib inline

def horizontal_flip_eye_data(df: pd.DataFrame, frame_width: int) -> pd.DataFrame:
    """
    Horizontally flip eye-tracking data across the vertical (y) axis.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns 'center_x', 'center_y', and 'phi' (in degrees).
    frame_width : int
        Width of the video/frame in pixels.

    Returns
    -------
    pd.DataFrame
        A copy of df where:
          - center_x → frame_width − center_x
          - center_y unchanged
          - phi      → (phi + 90) % 360
    """
    df_flipped = df.copy()
    # mirror x
    df_flipped['center_x'] = frame_width - df_flipped['center_x']
    # phi shift by +90°
    df_flipped['phi'] = (180 - df_flipped['phi']) % 360
    return df_flipped


def bokeh_plotter(data_list, x_axis_list=None, label_list=None,
                  plot_name='default',
                  x_axis_label='X', y_axis_label='Y',
                  peaks=None, peaks_list=False, export_path=False):
    """Generates an interactive Bokeh plot for the given data vector.
    Args:
        data_list (list or array): The data to be plotted.
        label_list (list of str): The labels of the data vectors
        plot_name (str, optional): The title of the plot. Defaults to 'default'.
        x_axis_label (str, optional): The label for the x-axis. Defaults to 'X'.
        y_axis_label (str, optional): The label for the y-axis. Defaults to 'Y'.
        peaks (list or array, optional): Indices of peaks to highlight on the plot. Defaults to None.
        export_path (False or str): when set to str, will output the resulting html fig
    """
    color_cycle = cycle(bokeh.palettes.Category10_10)
    fig = bokeh.plotting.figure(title=f'bokeh explorer: {plot_name}',
                                x_axis_label=x_axis_label,
                                y_axis_label=y_axis_label,
                                width=1500,
                                height=700)

    for i, data_vector in enumerate(data_list):
        color = next(color_cycle)
        data_vector = np.asarray(data_vector)

        if x_axis_list is None:
            x_axis = np.arange(len(data_vector))
        elif len(x_axis_list) == len(data_list):
            print('x_axis manually set')
            x_axis = np.asarray(x_axis_list[i])
        else:
            raise Exception(
                'problem with x_axis_list input - should be either None, or a list with the same length as data_list')
        
        if label_list is None:
            fig.line(x_axis, data_vector, line_color=color, legend_label=f"Line {i + 1}")
        elif len(label_list) == len(data_list):
            fig.line(x_axis, data_vector, line_color=color, legend_label=f"{label_list[i]}")
        
        if peaks is not None and peaks_list is True:
            if i < len(peaks):
                fig.scatter(peaks[i], data_vector[peaks[i]], size=10, color=color)

    if peaks is not None and peaks_list is False:
        if len(data_list) > 0:
            fig.scatter(peaks, data_vector[peaks], size=10, color='red')

    if export_path is not False:
        print(f'exporting to {export_path}')
        bokeh.io.output.output_file(filename=str(export_path / f'{plot_name}.html'), title=f'{plot_name}')
    bokeh.plotting.show(fig)


def load_eye_data(block):
    """
    Load the eye dataframes from CSV files created by the synchronization pipeline.
    No rotation matrices are loaded as rotation is no longer used.
    :param block: The current blocksync class
    :return: None
    """
    try:
        block.left_eye_data = pd.read_csv(block.analysis_path / 'left_eye_data.csv', index_col=0, engine='python')
        block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data.csv', index_col=0, engine='python')
        print(f'Loaded eye data for block {block.block_num}')
    except FileNotFoundError:
        print('Eye data files not found. Run the synchronization pipeline first!')
        raise


# create a multi-animal block_collection:

def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    """
    Create block collections and a block dictionary from multiple animals and their respective block lists.

    Parameters:
    - animals: list of str, names of the animals.
    - block_lists: list of lists of int, block numbers corresponding to each animal.
    - experiment_path: Path, path to the experiment directory.
    - bad_blocks: list of int, blocks to exclude. Default is an empty list.

    Returns:
    - block_collection: list of BlockSync objects for all specified blocks.
    - block_dict: dictionary where keys are block numbers as strings and values are BlockSync objects.
    """
    # uf is already imported at the top of the notebook

    if bad_blocks is None:
        bad_blocks = []

    block_collection = []
    block_dict = {}

    for animal, blocks in zip(animals, block_lists):
        # Generate blocks for the current animal
        current_blocks = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks
        )
        # Add to collection and dictionary
        block_collection.extend(current_blocks)
        for b in current_blocks:
            # Handle block_num as either int or string
            if isinstance(b.block_num, (int, float)):
                block_key = f"{animal}_block_{int(b.block_num):03d}"
            else:
                # If it's already a string, use it as-is
                block_key = f"{animal}_block_{b.block_num}"
            block_dict[block_key] = b

    return block_collection, block_dict


def load_self_kerr_refs(block, filename: str = "self_kerr_refs.csv") -> bool:
    """
    Load Kerr reference coordinates from the analysis folder CSV and set them on `block`.

    Reads a single-row CSV with columns:
        kerr_ref_r_x, kerr_ref_r_y, kerr_ref_l_x, kerr_ref_l_y

    Returns
    -------
    bool
        True if refs were loaded and applied, False if the file was missing or empty.
    """
    path = Path(block.analysis_path) / filename
    if not path.exists():
        print(f"No Kerr refs file found at: {path}")
        return False

    df = pd.read_csv(path)
    if df.empty:
        print(f"Kerr refs file is empty: {path}")
        return False

    row = df.iloc[0]

    # Helper to safely set attribute if value is finite
    def _set_attr(name):
        if name in row and pd.notna(row[name]):
            try:
                setattr(block, name, int(round(float(row[name]))))
            except (ValueError, TypeError):
                # keep existing value if conversion fails
                pass

    for col in ("kerr_ref_r_x", "kerr_ref_r_y", "kerr_ref_l_x", "kerr_ref_l_y"):
        _set_attr(col)

    print(f"Kerr refs loaded from: {path}")
    return True

In [ ]:
# Configure your experiment paths and blocks here
# Example:
# animals = ['PV_62', 'PV_126', 'PV_57']
# block_lists = [[24, 26, 38], [7, 8, 9, 10, 11, 12], [7, 8, 9, 12, 13]]
# experiment_path = Path(r"Z:\Nimrod\experiments")

animals = ['PV_106']
block_lists = [[15]]
experiment_path = Path(r"D:\sample_data_for_eye_repo")

bad_blocks = []  # Blocks to skip

block_collection, block_dict = create_block_collections(
    animals=animals,
    block_lists=block_lists,
    experiment_path=experiment_path,
    bad_blocks=bad_blocks
)

In [ ]:
# Load eye data from synchronization pipeline output
# Note: This assumes you've already run the synchronization and verification pipelines
for block in block_collection:
    # Only load the eye data - other steps should be done in previous pipelines
    try:
        load_eye_data(block)
    except FileNotFoundError:
        print(f'Warning: Eye data not found for block {block.block_num}. Run synchronization pipeline first.')
        continue



In [ ]:
for block in block_collection:
    load_self_kerr_refs(block)


In [ ]:
name_tag = 'raw_verified'
for block in block_collection:
    # Here is where the conversion happens:
    block.calculate_kerr_angles(name_tag=name_tag)


In [ ]:
# load and combine the eye data with the angle calculation
name_tag = 'raw_verified'
def append_angle_data(eye_df, new_df):
    """
    Appends the angle columns (phi and theta) from new_df to eye_df.
    The function renames 'phi' to 'k_phi' and 'theta' to 'k_theta', then merges
    on the shared 'OE_timestamp' column.

    Parameters:
    - eye_df: pandas DataFrame containing the eye tracking data.
    - new_df: pandas DataFrame containing the new kinematics data with columns
              'phi' and 'theta' along with 'OE_timestamp' (and possibly others).

    Returns:
    - merged_df: pandas DataFrame resulting from merging the new kinematics data
                 into eye_df.
    """
    # Select the necessary columns and rename them
    angle_data = new_df[['OE_timestamp', 'phi', 'theta']].rename(
        columns={'phi': 'k_phi', 'theta': 'k_theta'}
    )

    # Merge on OE_timestamp using a left join to preserve all rows in eye_df
    merged_df = pd.merge(eye_df, angle_data, on='OE_timestamp', how='left')

    return merged_df



for block in block_collection:
    print(block)
    try:
        left_angles = pd.read_csv([i for i in block.analysis_path.iterdir() if (f'left_kerr_angle_{name_tag}.csv' in str(i))][0])
        right_angles = pd.read_csv([i for i in block.analysis_path.iterdir() if (f'right_kerr_angle_{name_tag}.csv' in str(i))][0])
    except IndexError:
        print(f'{block} has a problem, files missing')

    block.left_eye_data = append_angle_data(block.left_eye_data,left_angles)
    block.right_eye_data = append_angle_data(block.right_eye_data,right_angles)



In [ ]:
def export_eye_data_w_angles(block, name_tag='0'):
    block.right_eye_data.to_csv(block.analysis_path / f'right_eye_data_{name_tag}.csv')
    block.left_eye_data.to_csv(block.analysis_path / f'left_eye_data_{name_tag}.csv')

for block in block_collection:
    export_eye_data_w_angles(block, name_tag='degrees_raw_verified')

In [ ]:
block_collection

In [ ]:
name_tag = 'degrees_raw_verified'
for block in block_collection:
    print(block.analysis_path / f'right_eye_data_{name_tag}.csv')
    block.right_eye_data.to_csv(block.analysis_path / f'right_eye_data_{name_tag}.csv')
    block.left_eye_data.to_csv(block.analysis_path / f'left_eye_data_{name_tag}.csv')